<div class="alert alert-block alert-info">

## Imports & Connection
</div>

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/fare-shock/policy_change'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/fare-shock/policy_change
Data dir -  /Users/E2074/local-datasets/customer/fare-shock/policy_change


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [5]:
## Parameter 
sep_start_date = '20240901'
sep_end_date = '20240929'
retention_start_date = '20240930'
retention_end_date = '20241110'

In [6]:
sdid_query = f"""
    SELECT 
        service.city_display_name AS city,
        service.service_level_l2 AS service_level_l2,
        service_level,
        service.service_detail_id AS service_detail_id
    FROM
        datasets.service_level_mapping_qc AS service
    WHERE 
        city_display_name IN ('bangalore', 'delhi')
        AND service_category in ('auto', 'cab')
        AND service_level_l2 in ('auto', 'cabeconomy')
        AND service_detail_id_isActive = true
    ORDER BY 1,2,3
    """
df_sdid = pd.read_sql(sdid_query, connection)

In [7]:
df_sdid

,city,service_level_l2,service_level,service_detail_id
0,bangalore,auto,auto,5c53562fceb6fc9241980547
1,bangalore,auto,cityauto,634c544443b7adfc2c5a4300
2,bangalore,cabeconomy,cabeconomy,6516a74cb0ef8abf16cec40d
3,delhi,auto,auto,5fc8c40bf4f0a70007d7cfbc
4,delhi,auto,auto ncr,64d22fba29b334580f1e3a18
5,delhi,cabeconomy,cabeconomy,648aaa1b4cf8d978be7ac5f6


In [8]:
sdid_list = df_sdid['service_detail_id'].tolist()
sdid_list

['5c53562fceb6fc9241980547',
 '634c544443b7adfc2c5a4300',
 '6516a74cb0ef8abf16cec40d',
 '5fc8c40bf4f0a70007d7cfbc',
 '64d22fba29b334580f1e3a18',
 '648aaa1b4cf8d978be7ac5f6']

<div class="alert alert-block alert-info">
    
### Data
</div>

In [51]:
def domain_fare_estimates_function(sep_start_date, sep_end_date, sdid_list, local_extract):

    df = pd.DataFrame()
    
    for sdid in sdid_list:
        
        base_query = f"""
        
        WITH domain_fare_estimates AS (
            SELECT
                yyyymmdd,
                customer_id,
                JSON_EXTRACT_SCALAR(service_obj, '$.estimates[0].serviceDetailId') servicedetailid,
                id,
                initial_estimated_fare,
                recalculated_fare,
                (recalculated_fare - initial_estimated_fare) difference,
                final_fare,
                -- recalculation_reason,
                -- fare_recalculated,
                -- recalculated_fare_used,
                data_set
            FROM
                iceberg.pricing.domain_fare_estimates_v3_immutable
            WHERE 
                yyyymmdd >= '{sep_start_date}'
                AND yyyymmdd <= '{sep_end_date}'
                AND recalculation_reason = 'routeDiff'
                AND fare_recalculated = 'true' 
                AND recalculated_fare_used = false
                AND JSON_EXTRACT_SCALAR(service_obj, '$.estimates[0].serviceDetailId') = '{sdid}'
            ),
            
            base AS (
            SELECT
                *
            FROM 
                domain_fare_estimates
            WHERE 
                difference < 0
            ),
            
            orders AS (
            SELECT 
                yyyymmdd,
                service_detail_id,
                customer_id,
                order_id,
                u.id_array as fare_estimate_id
            FROM 
                ( 
                SELECT 
                    yyyymmdd,
                    city_name,
                    service_obj_service_name service_name,
                    service_detail_id,
                    customer_id,
                    order_id,
                    fare_recalculated_reason,
                    case 
                        when fare_recalculated_type is null then 0 
                        when fare_recalculated_type = 'decrease' then fare_recalculated_diff_amount*-1
                        else fare_recalculated_diff_amount 
                    end fare_recalculated_diff_amount,
                    COALESCE(fare_recalculated_type, 'same') fare_recalculated_type,
                    amount,
                    estimate_id,
                    CAST(JSON_PARSE(estimate_ids) as ARRAY<VARCHAR>) AS id_array
                FROM 
                    orders.order_logs_snapshot ols  
                    
                WHERE 
                    yyyymmdd >= '{sep_start_date}'
                    AND yyyymmdd <= '{sep_end_date}'
                    AND order_status = 'dropped'
                    AND service_detail_id = '{sdid}'
                ) 
                AS t
                CROSS JOIN UNNEST(t.id_array) AS u(id_array)
            )
            
            
            SELECT 
                a.*,
                order_id
            FROM 
                base a
            LEFT JOIN 
                orders b
                ON a.yyyymmdd = b.yyyymmdd
                AND a.servicedetailid = b.service_detail_id
                AND a.customer_id = b.customer_id
                AND id = fare_estimate_id
        """
        
        df_base = pd.read_sql(base_query, connection)
        df = pd.concat([df, df_base], ignore_index=True)
        print('Data fetched successfully for sdid - ', sdid)
        df.to_parquet(local_extract + '/domain_fare_estimates.parquet', index=False)
        
    print('Data fetched successfully')
    
    return df

domain_fare_estimates_function(sep_start_date, sep_end_date, sdid_list, local_extract)

Data fetched successfully for sdid -  5c53562fceb6fc9241980547
Data fetched successfully for sdid -  634c544443b7adfc2c5a4300
Data fetched successfully for sdid -  6516a74cb0ef8abf16cec40d
Data fetched successfully for sdid -  5fc8c40bf4f0a70007d7cfbc
Data fetched successfully for sdid -  64d22fba29b334580f1e3a18
Data fetched successfully for sdid -  648aaa1b4cf8d978be7ac5f6
Data fetched successfully


,yyyymmdd,customer_id,servicedetailid,id,initial_estimated_fare,recalculated_fare,difference,final_fare,data_set,order_id
0,20240929,5dde36e3ee2b89270c78edf1,5c53562fceb6fc9241980547,66f9127432c5b8cfdb74058e,152,133,-19,152,"{""finalFare"":152,""finalLocationProvider"":""hF"",...",66f912881f7d80644bcf44d2
1,20240929,6469b79d161d3224dbe08ab3,5c53562fceb6fc9241980547,66f95efb2f8c41d35b987036,89,40,-49,89,"{""finalFare"":89,""finalLocationProvider"":""hF"",""...",66f95f5139608d3aeeb82c1d
2,20240929,65742a54cbd5973c356ea122,5c53562fceb6fc9241980547,66f902c844f1f5671712bcde,394,362,-32,394,"{""finalFare"":394,""finalLocationProvider"":""hF"",...",66f902e15fdbef0ebe877d79
3,20240929,6520be9f0d20f17162978501,5c53562fceb6fc9241980547,66f99187f133480c0f50b417,56,55,-1,56,"{""finalFare"":56,""finalLocationProvider"":""hF"",""...",66f991b1fb942e6ccde6d8c7
4,20240929,63d216ba3442951fe3181ba5,5c53562fceb6fc9241980547,66f9622ae4f811c01c22dd91,180,162,-18,181,"{""finalFare"":181,""finalLocationProvider"":""hF"",...",66f9623e39608d3aeeb833af
...,...,...,...,...,...,...,...,...,...,...
2264192,20240901,6631fa1f8a1f8d0b991be09c,648aaa1b4cf8d978be7ac5f6,66d45123f1f45291eb4c105f,243,57,-186,243,"{""finalFare"":243,""finalLocationProvider"":""Goog...",66d45123fac7540c582d8889
2264193,20240901,6468c3269cddd63aa3efddc0,648aaa1b4cf8d978be7ac5f6,66d452301dab827e1d5658c8,167,148,-19,167,"{""finalFare"":167,""finalLocationProvider"":""hF"",...",66d4523aae80b22dcb8529dc
2264194,20240901,61682af23cf262d59619a300,648aaa1b4cf8d978be7ac5f6,66d456effc366abb8f79afb7,236,215,-21,236,"{""finalFare"":236,""finalLocationProvider"":""hF"",...",66d4570b5eb53c1b3a76ca2d
2264195,20240901,658ad3d4124d6b349d064207,648aaa1b4cf8d978be7ac5f6,66d4598a744357352171231d,147,134,-13,147,"{""finalFare"":147,""finalLocationProvider"":""Goog...",66d4599ebc576064d6a45762


In [9]:
df_domain_fare_estimates = pd.read_parquet(local_extract + '/domain_fare_estimates.parquet')

In [10]:
df_domain_fare_estimates.shape

(2264197, 10)

In [11]:
df_domain_fare_estimates.head()

,yyyymmdd,customer_id,servicedetailid,id,initial_estimated_fare,recalculated_fare,difference,final_fare,data_set,order_id
0,20240929,5dde36e3ee2b89270c78edf1,5c53562fceb6fc9241980547,66f9127432c5b8cfdb74058e,152,133,-19,152,"{""finalFare"":152,""finalLocationProvider"":""hF"",...",66f912881f7d80644bcf44d2
1,20240929,6469b79d161d3224dbe08ab3,5c53562fceb6fc9241980547,66f95efb2f8c41d35b987036,89,40,-49,89,"{""finalFare"":89,""finalLocationProvider"":""hF"",""...",66f95f5139608d3aeeb82c1d
2,20240929,65742a54cbd5973c356ea122,5c53562fceb6fc9241980547,66f902c844f1f5671712bcde,394,362,-32,394,"{""finalFare"":394,""finalLocationProvider"":""hF"",...",66f902e15fdbef0ebe877d79
3,20240929,6520be9f0d20f17162978501,5c53562fceb6fc9241980547,66f99187f133480c0f50b417,56,55,-1,56,"{""finalFare"":56,""finalLocationProvider"":""hF"",""...",66f991b1fb942e6ccde6d8c7
4,20240929,63d216ba3442951fe3181ba5,5c53562fceb6fc9241980547,66f9622ae4f811c01c22dd91,180,162,-18,181,"{""finalFare"":181,""finalLocationProvider"":""hF"",...",66f9623e39608d3aeeb833af


In [12]:
df_domain_fare_estimates.servicedetailid.unique()

array(['5c53562fceb6fc9241980547', '6516a74cb0ef8abf16cec40d',
       '5fc8c40bf4f0a70007d7cfbc', '64d22fba29b334580f1e3a18',
       '648aaa1b4cf8d978be7ac5f6'], dtype=object)

In [13]:
def customer_order_query_function(sep_start_date, sep_end_date, sdid_list, local_extract):

    df = pd.DataFrame()
    
    for sdid in sdid_list:
        
        base_query = f"""
        
        WITH customer_orders AS (    
        
            -- SELECT 
            --     *,
            --     u.id_array as fare_estimate_id
            -- FROM 
                ( 
                SELECT 
                    yyyymmdd,
                    city_name,
                    service_obj_service_name service_name,
                    service_detail_id,
                    customer_id,
                    order_id,
                    fare_recalculated_reason,
                    case 
                        when fare_recalculated_type is null then 0 
                        when fare_recalculated_type = 'decrease' then fare_recalculated_diff_amount*-1
                        else fare_recalculated_diff_amount 
                    end fare_recalculated_diff_amount,
                    COALESCE(fare_recalculated_type, 'same') fare_recalculated_type,
                    amount,
                    estimate_id,
                    CAST(JSON_PARSE(estimate_ids) as ARRAY<VARCHAR>) AS id_array
                FROM 
                    orders.order_logs_snapshot ols  
                    
                WHERE 
                    yyyymmdd >= '{sep_start_date}'
                    AND yyyymmdd <= '{sep_end_date}'
                    AND order_status = 'dropped'
                    AND service_detail_id = '{sdid}'
                ) 
                -- AS t
                -- CROSS JOIN UNNEST(t.id_array) AS u(id_array)
            ),
        
            segment AS (
        
            SELECT 
                customer_id,
                taxi_retention_segments,
                taxi_regularity_segment,
                ps_tag_auto
            FROM 
                datasets.iallocator_customer_segments 
            WHERE 
                run_date = '2024-09-30'
                AND taxi_retention_segments != 'INACTIVE'
            )
            
            SELECT
                yyyymmdd,
                city_name,
                service_name,
                service_detail_id,
                customer_orders.customer_id,
                order_id,
                fare_recalculated_reason,
                fare_recalculated_diff_amount,
                fare_recalculated_type,
                amount,
                estimate_id,
                -- fare_estimate_id,
                coalesce(taxi_retention_segments, 'UNKNOWN') retention_segments,
                coalesce(taxi_regularity_segment, 'UNKNOWN') regularity_segments,
                coalesce(ps_tag_auto, 'UNKNOWN') ps_tag_auto 
            FROM 
                customer_orders
            LEFT JOIN 
                segment
                ON segment.customer_id = customer_orders.customer_id

            """
        df_base = pd.read_sql(base_query, connection)
        df = pd.concat([df, df_base], ignore_index=True)
        print('Data fetched successfully for sdid - ', sdid)
        df.to_parquet(local_extract + '/customer_order.parquet', index=False)
        
    print('Data fetched successfully')
    
    return df

customer_order_query_function(sep_start_date, sep_end_date, sdid_list, local_extract)

Data fetched successfully for sdid -  5c53562fceb6fc9241980547
Data fetched successfully for sdid -  634c544443b7adfc2c5a4300
Data fetched successfully for sdid -  6516a74cb0ef8abf16cec40d
Data fetched successfully for sdid -  5fc8c40bf4f0a70007d7cfbc
Data fetched successfully for sdid -  64d22fba29b334580f1e3a18
Data fetched successfully for sdid -  648aaa1b4cf8d978be7ac5f6
Data fetched successfully


,yyyymmdd,city_name,service_name,service_detail_id,customer_id,order_id,fare_recalculated_reason,fare_recalculated_diff_amount,fare_recalculated_type,amount,estimate_id,retention_segments,regularity_segments,ps_tag_auto
0,20240913,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,66e45101a6e3ce5f2dae7d1d,routeDiff,20,increase,123.0,66e45d498385ce8780b4bb8e,PRIME,WEEKLY,PS
1,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,66f6608111a6ef4e2004e703,None,0,same,310.0,66f660819e62b4674d17534a,PLATINUM,WEEKLY,UNKNOWN
2,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65841eced0269f8c95562d0e,66f663429e2a456423b58932,None,0,same,84.0,66f663002cf293416af76dbb,PLATINUM,WEEKLY,PS
3,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6331a989fcded19b614b061d,66f66bd15834435b3f11ed9f,routeDiff,0,same,56.0,66f66dc4afe48fcf9e119258,ELITE,DAILY,NPS
4,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6226c3e549fea91acf9f5fd0,66f66b539c002a5436eb885b,waitTimeCharges,3,increase,154.0,66f66b4b6f2c69bde5eee47f,ELITE,DAILY,NPS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11417368,20240912,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,6651ff2ad2a3e068db39c31c,66e324c0b392964657a5c3f4,None,0,same,149.0,66e324b7382109700b454f4b,ELITE,DAILY,NPS
11417369,20240916,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,668f6e03c2b70dcddf041ee0,66e7c4a70c18821b7f2d3e1a,None,0,same,206.0,66e7c4964c24fbdd0684118d,ELITE,DAILY,UNKNOWN
11417370,20240921,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,614803ddde864c824601edf9,66eeef5cab99a70aa3c2cf39,None,0,same,566.0,66eeef5ca48342ee499444d7,PLATINUM,WEEKLY,NPS
11417371,20240921,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,65ec66244b593aaea4a09936,66eef29fb39ef0352c4c12ac,dropDiff,0,decrease,302.0,66eef297d8983ee11a7d9b64,ELITE,WEEKLY,NPS


In [14]:
df_customer_order = pd.read_parquet(local_extract + '/customer_order.parquet')

In [15]:
df_customer_order.shape

(11417373, 14)

In [16]:
df_customer_order.head()

,yyyymmdd,city_name,service_name,service_detail_id,customer_id,order_id,fare_recalculated_reason,fare_recalculated_diff_amount,fare_recalculated_type,amount,estimate_id,retention_segments,regularity_segments,ps_tag_auto
0,20240913,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,66e45101a6e3ce5f2dae7d1d,routeDiff,20,increase,123.0,66e45d498385ce8780b4bb8e,PRIME,WEEKLY,PS
1,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,66f6608111a6ef4e2004e703,None,0,same,310.0,66f660819e62b4674d17534a,PLATINUM,WEEKLY,UNKNOWN
2,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65841eced0269f8c95562d0e,66f663429e2a456423b58932,None,0,same,84.0,66f663002cf293416af76dbb,PLATINUM,WEEKLY,PS
3,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6331a989fcded19b614b061d,66f66bd15834435b3f11ed9f,routeDiff,0,same,56.0,66f66dc4afe48fcf9e119258,ELITE,DAILY,NPS
4,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6226c3e549fea91acf9f5fd0,66f66b539c002a5436eb885b,waitTimeCharges,3,increase,154.0,66f66b4b6f2c69bde5eee47f,ELITE,DAILY,NPS


In [17]:
df_customer_order.city_name.unique()

array(['Bangalore', 'Delhi'], dtype=object)

In [47]:
def retension_customer_query_function(retention_start_date, retention_end_date, sdid_list, local_extract):

    df = pd.DataFrame()
    
    for sdid in sdid_list:
        
        base_query = f"""
        
        SELECT 
            DISTINCT
            date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
            city_name,
            service_obj_service_name service_name,
            service_detail_id,
            customer_id
        FROM 
            orders.order_logs_snapshot ols
        WHERE 
            yyyymmdd >= '{retention_start_date}'
            AND yyyymmdd <= '{retention_end_date}'
            AND order_status = 'dropped'
            AND service_detail_id = '{sdid}'

            """
        df_base = pd.read_sql(base_query, connection)
        df = pd.concat([df, df_base], ignore_index=True)
        print('Data fetched successfully for sdid - ', sdid)
        df.to_parquet(local_extract + '/retension_customer.parquet', index=False)
        
    print('Data fetched successfully')
    
    return df

retension_customer_query_function(retention_start_date, retention_end_date, sdid_list, local_extract)

Data fetched successfully for sdid -  5c53562fceb6fc9241980547
Data fetched successfully for sdid -  634c544443b7adfc2c5a4300
Data fetched successfully for sdid -  6516a74cb0ef8abf16cec40d
Data fetched successfully for sdid -  5fc8c40bf4f0a70007d7cfbc
Data fetched successfully for sdid -  64d22fba29b334580f1e3a18
Data fetched successfully for sdid -  648aaa1b4cf8d978be7ac5f6
Data fetched successfully


,week_start_date,city_name,service_name,service_detail_id,customer_id
0,20241104,Bangalore,Auto,5c53562fceb6fc9241980547,636791add58af7606ae3cc89
1,20241028,Bangalore,Auto,5c53562fceb6fc9241980547,669f4349968844374aff8541
2,20240930,Bangalore,Auto,5c53562fceb6fc9241980547,62824533d7dc68b90a756e83
3,20241104,Bangalore,Auto,5c53562fceb6fc9241980547,62a955a25948ea5765054177
4,20241104,Bangalore,Auto,5c53562fceb6fc9241980547,65fa6135a6458534260572e7
...,...,...,...,...,...
9382918,20241007,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,659257857a56fef292cf1017
9382919,20241007,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,6606f403d70f5db2f9a252be
9382920,20241007,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,65baedf9276cb44bc43d90d2
9382921,20241007,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,64c2953e0b4d07aa9f1ad927


In [18]:
df_retension_customer = pd.read_parquet(local_extract + '/retension_customer.parquet')

In [19]:
df_retension_customer.shape

(9382923, 5)

In [20]:
df_customer_order.head()

,yyyymmdd,city_name,service_name,service_detail_id,customer_id,order_id,fare_recalculated_reason,fare_recalculated_diff_amount,fare_recalculated_type,amount,estimate_id,retention_segments,regularity_segments,ps_tag_auto
0,20240913,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,66e45101a6e3ce5f2dae7d1d,routeDiff,20,increase,123.0,66e45d498385ce8780b4bb8e,PRIME,WEEKLY,PS
1,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,66f6608111a6ef4e2004e703,None,0,same,310.0,66f660819e62b4674d17534a,PLATINUM,WEEKLY,UNKNOWN
2,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65841eced0269f8c95562d0e,66f663429e2a456423b58932,None,0,same,84.0,66f663002cf293416af76dbb,PLATINUM,WEEKLY,PS
3,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6331a989fcded19b614b061d,66f66bd15834435b3f11ed9f,routeDiff,0,same,56.0,66f66dc4afe48fcf9e119258,ELITE,DAILY,NPS
4,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6226c3e549fea91acf9f5fd0,66f66b539c002a5436eb885b,waitTimeCharges,3,increase,154.0,66f66b4b6f2c69bde5eee47f,ELITE,DAILY,NPS


In [21]:
df_customer_order.city_name.unique()

array(['Bangalore', 'Delhi'], dtype=object)

<div class="alert alert-block alert-info">
    
## Data Processing
</div>

In [22]:
df_domain_fare_estimates.head()

,yyyymmdd,customer_id,servicedetailid,id,initial_estimated_fare,recalculated_fare,difference,final_fare,data_set,order_id
0,20240929,5dde36e3ee2b89270c78edf1,5c53562fceb6fc9241980547,66f9127432c5b8cfdb74058e,152,133,-19,152,"{""finalFare"":152,""finalLocationProvider"":""hF"",...",66f912881f7d80644bcf44d2
1,20240929,6469b79d161d3224dbe08ab3,5c53562fceb6fc9241980547,66f95efb2f8c41d35b987036,89,40,-49,89,"{""finalFare"":89,""finalLocationProvider"":""hF"",""...",66f95f5139608d3aeeb82c1d
2,20240929,65742a54cbd5973c356ea122,5c53562fceb6fc9241980547,66f902c844f1f5671712bcde,394,362,-32,394,"{""finalFare"":394,""finalLocationProvider"":""hF"",...",66f902e15fdbef0ebe877d79
3,20240929,6520be9f0d20f17162978501,5c53562fceb6fc9241980547,66f99187f133480c0f50b417,56,55,-1,56,"{""finalFare"":56,""finalLocationProvider"":""hF"",""...",66f991b1fb942e6ccde6d8c7
4,20240929,63d216ba3442951fe3181ba5,5c53562fceb6fc9241980547,66f9622ae4f811c01c22dd91,180,162,-18,181,"{""finalFare"":181,""finalLocationProvider"":""hF"",...",66f9623e39608d3aeeb833af


In [23]:
customer_negative_bins = df_domain_fare_estimates[df_domain_fare_estimates['difference'] < 0]\
                            .groupby(['servicedetailid', 'customer_id' ])\
                            .agg(negative_order_bin = ('order_id', 'nunique'))\
                            .reset_index()
customer_negative_bins.head()

,servicedetailid,customer_id,negative_order_bin
0,5c53562fceb6fc9241980547,5737c6d0ddbec2203f73329f,1
1,5c53562fceb6fc9241980547,5737c6e3ddbec2203f733341,1
2,5c53562fceb6fc9241980547,5737c6f3ddbec2203f7333d1,3
3,5c53562fceb6fc9241980547,5737c6f6ddbec2203f7333e9,1
4,5c53562fceb6fc9241980547,5737c70eddbec2203f7334b8,2


In [24]:
df_customer_order.head()

,yyyymmdd,city_name,service_name,service_detail_id,customer_id,order_id,fare_recalculated_reason,fare_recalculated_diff_amount,fare_recalculated_type,amount,estimate_id,retention_segments,regularity_segments,ps_tag_auto
0,20240913,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,66e45101a6e3ce5f2dae7d1d,routeDiff,20,increase,123.0,66e45d498385ce8780b4bb8e,PRIME,WEEKLY,PS
1,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,66f6608111a6ef4e2004e703,None,0,same,310.0,66f660819e62b4674d17534a,PLATINUM,WEEKLY,UNKNOWN
2,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65841eced0269f8c95562d0e,66f663429e2a456423b58932,None,0,same,84.0,66f663002cf293416af76dbb,PLATINUM,WEEKLY,PS
3,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6331a989fcded19b614b061d,66f66bd15834435b3f11ed9f,routeDiff,0,same,56.0,66f66dc4afe48fcf9e119258,ELITE,DAILY,NPS
4,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6226c3e549fea91acf9f5fd0,66f66b539c002a5436eb885b,waitTimeCharges,3,increase,154.0,66f66b4b6f2c69bde5eee47f,ELITE,DAILY,NPS


In [25]:
# Merge 
df_merge_bin = pd.merge(df_customer_order,
                        customer_negative_bins,
                        how='left',
                        left_on=['service_detail_id', 'customer_id'],
                        right_on=['servicedetailid', 'customer_id']
                        )
df_merge_bin['negative_order_bin'] = df_merge_bin['negative_order_bin'].fillna(0)

In [26]:
df_merge_bin.head()

,yyyymmdd,city_name,service_name,service_detail_id,customer_id,order_id,fare_recalculated_reason,fare_recalculated_diff_amount,fare_recalculated_type,amount,estimate_id,retention_segments,regularity_segments,ps_tag_auto,servicedetailid,negative_order_bin
0,20240913,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,66e45101a6e3ce5f2dae7d1d,routeDiff,20,increase,123.0,66e45d498385ce8780b4bb8e,PRIME,WEEKLY,PS,5c53562fceb6fc9241980547,2.0
1,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,66f6608111a6ef4e2004e703,None,0,same,310.0,66f660819e62b4674d17534a,PLATINUM,WEEKLY,UNKNOWN,NaN,0.0
2,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65841eced0269f8c95562d0e,66f663429e2a456423b58932,None,0,same,84.0,66f663002cf293416af76dbb,PLATINUM,WEEKLY,PS,5c53562fceb6fc9241980547,2.0
3,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6331a989fcded19b614b061d,66f66bd15834435b3f11ed9f,routeDiff,0,same,56.0,66f66dc4afe48fcf9e119258,ELITE,DAILY,NPS,5c53562fceb6fc9241980547,1.0
4,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6226c3e549fea91acf9f5fd0,66f66b539c002a5436eb885b,waitTimeCharges,3,increase,154.0,66f66b4b6f2c69bde5eee47f,ELITE,DAILY,NPS,NaN,0.0


In [27]:
# Validation at city | service | customer level

df_merge_bin\
.groupby(['city_name', 'service_name', 'service_detail_id', 'customer_id', 'regularity_segments'])\
.agg(counts = ('negative_order_bin', 'nunique'))\
.sort_values(by=['counts'], ascending=False)\
.reset_index()\
.head()

,city_name,service_name,service_detail_id,customer_id,regularity_segments,counts
0,Bangalore,Auto,5c53562fceb6fc9241980547,5737c6d0ddbec2203f73329f,BI_WEEKLY,1
1,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,63233a9b5d3af4d950021a03,BI_WEEKLY,1
2,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,6323303e0f4eef94222e4c8f,DAILY,1
3,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,632330555ff6d3be234edcc7,WEEKLY,1
4,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,6323316a01c2fbb816ca5acd,BI_WEEKLY,1


In [28]:
df_customer_tag = df_merge_bin[['city_name', 'service_name', 'service_detail_id', 
                                'customer_id', 'regularity_segments', 'ps_tag_auto', 'negative_order_bin']]

<div class="alert alert-block alert-success">
    
### Analysis
</div>

In [29]:
df_customer_tag.head(1)

,city_name,service_name,service_detail_id,customer_id,regularity_segments,ps_tag_auto,negative_order_bin
0,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,WEEKLY,PS,2.0


In [30]:
bins = [0, 1, 2, 5, float('inf')]  # float('inf') is used for 15+
labels = ['0', '1', '2-4', '5+']

# Create a new column with the bins
df_customer_tag['bins'] = pd.cut(df_customer_tag['negative_order_bin'], bins=bins, labels=labels, right=False)

In [31]:
df_retension_customer.head(1)

,week_start_date,city_name,service_name,service_detail_id,customer_id
0,20241104,Bangalore,Auto,5c53562fceb6fc9241980547,636791add58af7606ae3cc89


In [32]:
retention_df = pd.merge(df_customer_tag,
                        df_retension_customer,
                        how='left',
                        left_on=['city_name', 'service_name', 'service_detail_id', 'customer_id'],
                        right_on=['city_name', 'service_name', 'service_detail_id', 'customer_id']
                        )

In [33]:
retention_df.shape

(33118318, 9)

In [34]:
retention_df

,city_name,service_name,service_detail_id,customer_id,regularity_segments,ps_tag_auto,negative_order_bin,bins,week_start_date
0,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,WEEKLY,PS,2.0,2-4,20241014
1,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,WEEKLY,UNKNOWN,0.0,0,20241021
2,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,WEEKLY,UNKNOWN,0.0,0,20241007
3,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,WEEKLY,UNKNOWN,0.0,0,20241028
4,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,WEEKLY,UNKNOWN,0.0,0,20241014
...,...,...,...,...,...,...,...,...,...
33118313,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,668f6e03c2b70dcddf041ee0,DAILY,UNKNOWN,0.0,0,20241014
33118314,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,614803ddde864c824601edf9,WEEKLY,NPS,0.0,0,20241014
33118315,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,614803ddde864c824601edf9,WEEKLY,NPS,0.0,0,20241104
33118316,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,65ec66244b593aaea4a09936,WEEKLY,NPS,0.0,0,NaN


In [35]:
df_grouped = retention_df\
                .groupby(['city_name', 'service_name', 'service_detail_id', 'ps_tag_auto', 'bins'])\
                .agg(unique_customers=('customer_id', 'nunique'))\
                .reset_index()
df_grouped.head(2)

,city_name,service_name,service_detail_id,ps_tag_auto,bins,unique_customers
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,343648
1,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,1,203391


In [36]:
df_grouped_weekly = retention_df\
                        .groupby(['city_name', 'service_name', 'service_detail_id', 'ps_tag_auto', 
                                  'bins', 'week_start_date'])\
                        .agg(unique_customers=('customer_id', 'nunique'))\
                        .reset_index()
df_grouped_weekly.head(2)

,city_name,service_name,service_detail_id,ps_tag_auto,bins,week_start_date,unique_customers
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,20240930,107136
1,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,20241007,102427


In [37]:
df_pivot_table = df_grouped_weekly.pivot(index=['city_name', 'service_name', 'service_detail_id', 
                                                'ps_tag_auto', 'bins'],
                         columns='week_start_date',
                         values='unique_customers',
                        ).reset_index()
pivot_table = df_pivot_table.reset_index(drop=True)
pivot_table

week_start_date,city_name,service_name,service_detail_id,ps_tag_auto,bins,20240930,20241007,20241014,20241021,20241028,20241104
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,107136,102427,109681,99391,88820,94732
1,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,1,74452,71009,73594,67819,60764,64539
2,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,2-4,69805,65009,65670,62441,54934,58468
3,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,5+,28605,26362,26369,25788,22425,24060
4,Bangalore,Auto,5c53562fceb6fc9241980547,PS,0,59029,56939,62344,54165,48752,50734
...,...,...,...,...,...,...,...,...,...,...,...
355,Delhi,CabEconomy,6516a74cb0ef8abf16cec40d,PS,5+,0,0,0,0,0,0
356,Delhi,CabEconomy,6516a74cb0ef8abf16cec40d,UNKNOWN,0,0,0,0,0,0,0
357,Delhi,CabEconomy,6516a74cb0ef8abf16cec40d,UNKNOWN,1,0,0,0,0,0,0
358,Delhi,CabEconomy,6516a74cb0ef8abf16cec40d,UNKNOWN,2-4,0,0,0,0,0,0


<div class="alert alert-block alert-success">
    
### Retention %
</div>

In [38]:
df_grouped.shape

(360, 6)

In [39]:
pivot_table.shape

(360, 11)

In [40]:
df_grouped.head(1)

,city_name,service_name,service_detail_id,ps_tag_auto,bins,unique_customers
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,343648


In [41]:
pivot_table.head(1)

week_start_date,city_name,service_name,service_detail_id,ps_tag_auto,bins,20240930,20241007,20241014,20241021,20241028,20241104
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,107136,102427,109681,99391,88820,94732


In [42]:
df_final = pd.merge(df_grouped,
                    pivot_table,
                    how='inner',
                    on=['city_name', 'service_name', 'service_detail_id', 'ps_tag_auto', 'bins']
                   )
df_final.head(2)

,city_name,service_name,service_detail_id,ps_tag_auto,bins,unique_customers,20240930,20241007,20241014,20241021,20241028,20241104
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,343648,107136,102427,109681,99391,88820,94732
1,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,1,203391,74452,71009,73594,67819,60764,64539


In [43]:
date_columns = ["20240930", "20241007", "20241014", "20241021", "20241028", "20241104"]

for date in date_columns:

    df_final[f"{date}_%"] = (df_final[date]*100.00/df_final["unique_customers"]).round(2)

In [44]:
df_final = df_final[df_final['unique_customers'] > 0]

In [45]:
df_final

,city_name,service_name,service_detail_id,ps_tag_auto,bins,unique_customers,20240930,20241007,20241014,20241021,20241028,20241104,20240930_%,20241007_%,20241014_%,20241021_%,20241028_%,20241104_%
0,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,0,343648,107136,102427,109681,99391,88820,94732,31.18,29.81,31.92,28.92,25.85,27.57
1,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,1,203391,74452,71009,73594,67819,60764,64539,36.61,34.91,36.18,33.34,29.88,31.73
2,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,2-4,127219,69805,65009,65670,62441,54934,58468,54.87,51.10,51.62,49.08,43.18,45.96
3,Bangalore,Auto,5c53562fceb6fc9241980547,NPS,5+,34754,28605,26362,26369,25788,22425,24060,82.31,75.85,75.87,74.20,64.52,69.23
4,Bangalore,Auto,5c53562fceb6fc9241980547,PS,0,212178,59029,56939,62344,54165,48752,50734,27.82,26.84,29.38,25.53,22.98,23.91
5,Bangalore,Auto,5c53562fceb6fc9241980547,PS,1,137034,43575,41958,44484,39272,35575,37019,31.80,30.62,32.46,28.66,25.96,27.01
6,Bangalore,Auto,5c53562fceb6fc9241980547,PS,2-4,76186,38279,35647,36352,33655,29870,31477,50.24,46.79,47.71,44.17,39.21,41.32
7,Bangalore,Auto,5c53562fceb6fc9241980547,PS,5+,17608,14102,13113,12938,12646,10900,11743,80.09,74.47,73.48,71.82,61.90,66.69
8,Bangalore,Auto,5c53562fceb6fc9241980547,UNKNOWN,0,330998,52082,49609,54606,48498,43501,46185,15.73,14.99,16.50,14.65,13.14,13.95
9,Bangalore,Auto,5c53562fceb6fc9241980547,UNKNOWN,1,139066,22816,21961,23398,20806,19001,19937,16.41,15.79,16.83,14.96,13.66,14.34


In [46]:
df_final[['city_name','service_name', 'ps_tag_auto', 'bins',
          'unique_customers', 
          '20240930_%',	'20241007_%',	'20241014_%',	'20241021_%', '20241028_%',	'20241104_%' ]]

,city_name,service_name,ps_tag_auto,bins,unique_customers,20240930_%,20241007_%,20241014_%,20241021_%,20241028_%,20241104_%
0,Bangalore,Auto,NPS,0,343648,31.18,29.81,31.92,28.92,25.85,27.57
1,Bangalore,Auto,NPS,1,203391,36.61,34.91,36.18,33.34,29.88,31.73
2,Bangalore,Auto,NPS,2-4,127219,54.87,51.10,51.62,49.08,43.18,45.96
3,Bangalore,Auto,NPS,5+,34754,82.31,75.85,75.87,74.20,64.52,69.23
4,Bangalore,Auto,PS,0,212178,27.82,26.84,29.38,25.53,22.98,23.91
5,Bangalore,Auto,PS,1,137034,31.80,30.62,32.46,28.66,25.96,27.01
6,Bangalore,Auto,PS,2-4,76186,50.24,46.79,47.71,44.17,39.21,41.32
7,Bangalore,Auto,PS,5+,17608,80.09,74.47,73.48,71.82,61.90,66.69
8,Bangalore,Auto,UNKNOWN,0,330998,15.73,14.99,16.50,14.65,13.14,13.95
9,Bangalore,Auto,UNKNOWN,1,139066,16.41,15.79,16.83,14.96,13.66,14.34


In [56]:
df_final.to_clipboard(index=False)

### Orders

In [47]:
df_merge_bin.head()

,yyyymmdd,city_name,service_name,service_detail_id,customer_id,order_id,fare_recalculated_reason,fare_recalculated_diff_amount,fare_recalculated_type,amount,estimate_id,retention_segments,regularity_segments,ps_tag_auto,servicedetailid,negative_order_bin
0,20240913,Bangalore,Auto,5c53562fceb6fc9241980547,590b939933c57ee92c8a78ff,66e45101a6e3ce5f2dae7d1d,routeDiff,20,increase,123.0,66e45d498385ce8780b4bb8e,PRIME,WEEKLY,PS,5c53562fceb6fc9241980547,2.0
1,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65b8126fec61531661fc53d3,66f6608111a6ef4e2004e703,None,0,same,310.0,66f660819e62b4674d17534a,PLATINUM,WEEKLY,UNKNOWN,NaN,0.0
2,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,65841eced0269f8c95562d0e,66f663429e2a456423b58932,None,0,same,84.0,66f663002cf293416af76dbb,PLATINUM,WEEKLY,PS,5c53562fceb6fc9241980547,2.0
3,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6331a989fcded19b614b061d,66f66bd15834435b3f11ed9f,routeDiff,0,same,56.0,66f66dc4afe48fcf9e119258,ELITE,DAILY,NPS,5c53562fceb6fc9241980547,1.0
4,20240927,Bangalore,Auto,5c53562fceb6fc9241980547,6226c3e549fea91acf9f5fd0,66f66b539c002a5436eb885b,waitTimeCharges,3,increase,154.0,66f66b4b6f2c69bde5eee47f,ELITE,DAILY,NPS,NaN,0.0


In [48]:
df_test_orders = df_merge_bin.copy()

In [49]:
bins_test = [0, 1, float('inf')]  # float('inf') is used for 15+
labels_test = ['Zero', '1 =< ' ]

# Create a new column with the bins
df_test_orders['bins'] = pd.cut(df_test_orders['negative_order_bin'], bins=bins_test, labels=labels_test, right=False)

In [50]:
df1 = df_test_orders\
.groupby(['city_name', 'service_name', 'service_detail_id', 'ps_tag_auto', 'bins'])\
.agg(orders = ('order_id', 'nunique'))\
.reset_index()

In [51]:
df1 = df1[df1['orders'] > 0]

In [59]:
df1.to_clipboard(index=False)

In [52]:
df2 = df_test_orders\
.groupby(['city_name', 'service_name', 'service_detail_id', 'bins'])\
.agg(orders = ('order_id', 'nunique'))\
.reset_index()

In [53]:
df2 = df2[df2['orders'] > 0]

In [54]:
df2

,city_name,service_name,service_detail_id,bins,orders
0,Bangalore,Auto,5c53562fceb6fc9241980547,Zero,1964424
1,Bangalore,Auto,5c53562fceb6fc9241980547,1 =<,4369488
28,Bangalore,CabEconomy,6516a74cb0ef8abf16cec40d,Zero,544698
29,Bangalore,CabEconomy,6516a74cb0ef8abf16cec40d,1 =<,409216
32,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,Zero,1059465
33,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,1 =<,1318029
46,Delhi,Auto NCR,64d22fba29b334580f1e3a18,Zero,42776
47,Delhi,Auto NCR,64d22fba29b334580f1e3a18,1 =<,44094
54,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,Zero,1425298
55,Delhi,CabEconomy,648aaa1b4cf8d978be7ac5f6,1 =<,239885


In [62]:
df2.to_clipboard(index=False)